In [ ]:
!pip install torch_geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split
import torch_geometric
from torch_geometric.loader import DataLoader
from torch_geometric.nn import PNAConv, global_add_pool
from torch_geometric.utils import degree
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Mounted at /content/drive
Using device: cuda


In [ ]:
import os
BASE_PATH  = '/content/drive/MyDrive/team3_xai_gnn'
DATA_DIR   = os.path.join(BASE_PATH, 'preprocessed')
OUTPUT_DIR = os.path.join(BASE_PATH, 'pna_results')
INNER_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'inner_results.pkl')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# PNA (Principal Neighbourhood Aggregation) Model Definition
# This class defines the architecture of the PNA Graph Neural Network.
class PNANet(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, aggregators, scalers, deg, edge_dim):
        super(PNANet, self).__init__()
        # nn.ModuleList to hold multiple PNAConv layers, BatchNorm layers, and Dropout layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropouts = nn.ModuleList()

        # Initial PNAConv layer:
        #   - in_channels: Dimension of input node features
        #   - hidden_channels: Dimension of output node features for this l
        `ayer
        #   - aggregators: List of aggregation functions (e.g., mean, max, std)
        #   - scalers: List of scaling functions (e.g., identity, amplification, attenuation)
        #   - deg: Degree distribution histogram from the training set, used for degree-aware scaling
        #   - edge_dim: Dimension of edge features, to be incorporated into aggregation
        self.convs.append(PNAConv(in_channels, hidden_channels, aggregators=aggregators, scalers=scalers, deg=deg, edge_dim=edge_dim))
        # Batch normalization layer for stabilizing and accelerating training
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        # Dropout layer for regularization to prevent overfitting
        self.dropouts.append(nn.Dropout(0.5)) # Dropout rate of 50%

        # Subsequent PNAConv layers (num_layers - 1 times):
        # These layers maintain the hidden_channels dimension
        for _ in range(num_layers - 1):
            self.convs.append(PNAConv(hidden_channels, hidden_channels, aggregators=aggregators, scalers=scalers, deg=deg, edge_dim=edge_dim))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            self.dropouts.append(nn.Dropout(0.5))

        # Output head (linear layer):
        # Maps the final hidden dimension to the desired number of output classes (out_channels)
        self.lin = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_attr, batch):
        # Forward pass through each PNAConv layer
        for conv, bn, dropout in zip(self.convs, self.bns, self.dropouts):
            x = conv(x, edge_index, edge_attr) # Apply PNA convolution
            x = bn(x)                          # Apply batch normalization
            x = F.relu(x)                      # Apply ReLU activation function
            x = dropout(x)                     # Apply dropout

        # Global pooling (e.g., global_add_pool) after GNN layers:
        # This aggregates node features into a single graph-level representation.
        # 'batch' tensor specifies to which graph each node belongs.
        x = global_add_pool(x, batch)

        # Final linear layer for classification
        x = self.lin(x)
        return x # Return logits, CrossEntropyLoss expects logits

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        # --- step 1
        # standard cross entropy, one value per sample (no reduction yet)
        # this gives us -log(pt) for each sample
        ce_loss = nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            reduction='none'   # keep per-sample losses, don't average yet
        )

        # --- step 2
        # recover pt (the probability assigned to the correct class)
        # ce_loss = -log(pt)  →  pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)

        # --- step 3
        # apply the focal modulating factor (1 - pt)^gamma
        # when pt is high (easy example, model is confident) → factor is small → loss shrinks
        # when pt is low  (hard example, model is uncertain) → factor is ~1   → loss unchanged
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()

In [ ]:
# helper functions
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils import compute_class_weight
import copy # Ensure copy is imported for deepcopy

def get_labels(dataset):
    return np.array([d.y.item() for d in dataset])


def compute_metrics(labels, preds, probs):
    return {
        'balanced_acc':  balanced_accuracy_score(labels, preds),
        'accuracy':      accuracy_score(labels, preds),
        'recall_mci':    recall_score(labels, preds, pos_label=1, zero_division=0),
        'recall_cn':     recall_score(labels, preds, pos_label=0, zero_division=0),
        'precision_mci': precision_score(labels, preds, pos_label=1, zero_division=0),
        'precision_cn':  precision_score(labels, preds, pos_label=0, zero_division=0),
        'f1_mci':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'f1_cn':         f1_score(labels, preds, pos_label=0, zero_division=0),
        'f1_macro':      f1_score(labels, preds, average='macro', zero_division=0),
        'auc':           roc_auc_score(labels, probs[:, 1]) if len(np.unique(labels)) > 1 else float('nan'),
    }


def avg_metrics(metrics_list):
    keys = metrics_list[0].keys()
    return {k: float(np.mean([m[k] for m in metrics_list])) for k in keys}


def passes_filter(avg, thresholds):
    return all(avg.get(k, 0.0) >= v for k, v in thresholds.items())


def build_criterion(cfg, train_labels):
    cw    = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
    cw[1] = cfg['mci_weight']
    return FocalLoss(
        gamma=cfg['gamma'],
        weight=torch.tensor(cw, dtype=torch.float).to(device)
    )


def build_model_and_opt(cfg, deg): # Added 'deg' as an argument
    # Fixed aggregators and scalers for now to manage hyperparameter space.
    # These could be tuned if more combinations were allowed.
    aggregators = ['mean', 'max', 'std']
    scalers = ['identity', 'amplification']

    # Instantiate PNANet instead of TransformerConvNet
    model = PNANet(
        in_channels=NODE_IN,
        hidden_channels=HIDDEN,
        out_channels=2, # Assuming binary classification (CN vs MCI)
        num_layers=cfg['num_layers'], # num_layers now comes from cfg
        aggregators=aggregators,
        scalers=scalers,
        deg=deg, # Pass the calculated degree distribution
        edge_dim=EDGE_IN
    ).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=cfg['wd'])
    return model, opt


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss   = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_labels.append(batch.y.cpu().numpy())
        all_preds.append(probs.argmax(axis=1))
        all_probs.append(probs)
    return (
        np.concatenate(all_labels),
        np.concatenate(all_preds),
        np.concatenate(all_probs, axis=0),
    )


def train_with_early_stopping(model, opt, criterion, train_loader, val_loader, deg): # Added 'deg' as an argument
    best_bal_acc   = -1.0
    best_state     = None
    best_metrics   = None
    patience_count = 0

    for epoch in range(EPOCHS):
        train_one_epoch(model, train_loader, criterion, opt)
        labels, preds, probs = evaluate(model, val_loader)
        metrics = compute_metrics(labels, preds, probs)

        if metrics['balanced_acc'] > best_bal_acc:
            best_bal_acc   = metrics['balanced_acc']
            best_state     = copy.deepcopy(model.state_dict())
            best_metrics   = metrics
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_metrics

In [ ]:
import itertools

# fixed params
NODE_IN  = 4
EDGE_IN  = 15
HIDDEN   = 32
HEADS    = 2 # Note: HEADS is not used in PNANet, but kept here for historical context if needed.
DROPOUT  = 0.3
LR       = 1e-3
EPOCHS   = 50
PATIENCE = 15
BATCH    = 32
# k-folds
N_OUTER  = 5
N_INNER  = 3

# hyperparams for tuning
gammas        = [1.0, 1.5, 2.0]
mci_weights   = [3.5, 4.0, 4.5]
weight_decays = [1e-3, 1e-4]
num_layers    = [2, 3] # New hyperparameter for PNANet

CONFIGS = [
    {'mci_weight': mw, 'gamma': g, 'wd': wd, 'num_layers': nl}
    for mw, g, wd, nl in itertools.product(mci_weights, gammas, weight_decays, num_layers)
]

print(f'Total configs: {len(CONFIGS)}')
for i, c in enumerate(CONFIGS):
    print(f'  [{i:02d}] {c}')

Total configs: 36
  [00] {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.001, 'num_layers': 2}
  [01] {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.001, 'num_layers': 3}
  [02] {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.0001, 'num_layers': 2}
  [03] {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.0001, 'num_layers': 3}
  [04] {'mci_weight': 3.5, 'gamma': 1.5, 'wd': 0.001, 'num_layers': 2}
  [05] {'mci_weight': 3.5, 'gamma': 1.5, 'wd': 0.001, 'num_layers': 3}
  [06] {'mci_weight': 3.5, 'gamma': 1.5, 'wd': 0.0001, 'num_layers': 2}
  [07] {'mci_weight': 3.5, 'gamma': 1.5, 'wd': 0.0001, 'num_layers': 3}
  [08] {'mci_weight': 3.5, 'gamma': 2.0, 'wd': 0.001, 'num_layers': 2}
  [09] {'mci_weight': 3.5, 'gamma': 2.0, 'wd': 0.001, 'num_layers': 3}
  [10] {'mci_weight': 3.5, 'gamma': 2.0, 'wd': 0.0001, 'num_layers': 2}
  [11] {'mci_weight': 3.5, 'gamma': 2.0, 'wd': 0.0001, 'num_layers': 3}
  [12] {'mci_weight': 4.0, 'gamma': 1.0, 'wd': 0.001, 'num_layers': 2}
  [13] {'mci_weight': 4.0, 'gamma': 1.0, 'wd': 0.001,

In [ ]:
import pickle
import os
import torch

# data loading
print('Loading data...')

outer_folds = []
for k in range(1, N_OUTER + 1):
    outer_folds.append({
        'train': torch.load(os.path.join(DATA_DIR, f'fold_{k}_train.pt'), weights_only=False),
        'val':   torch.load(os.path.join(DATA_DIR, f'fold_{k}_val.pt'), weights_only=False),
        'test':  torch.load(os.path.join(DATA_DIR, f'fold_{k}_test.pt'), weights_only=False),
    })

with open(os.path.join(DATA_DIR, 'inner_fold_ids.pkl'), 'rb') as f:
    inner_fold_ids = pickle.load(f)

print('Done.')
for k, fold in enumerate(outer_folds):
    print(f'  Outer fold {k+1}: train={len(fold["train"])}  val={len(fold["val"])}  test={len(fold["test"])}')

Loading data...
Done.
  Outer fold 1: train=431  val=77  test=127
  Outer fold 2: train=431  val=77  test=127
  Outer fold 3: train=431  val=77  test=127
  Outer fold 4: train=431  val=77  test=127
  Outer fold 5: train=431  val=77  test=127


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torch_geometric.utils._scatter')
import pandas as pd

# resume if training stopped midway
if os.path.exists(INNER_RESULTS_PATH):
    with open(INNER_RESULTS_PATH, 'rb') as f:
        inner_results = pickle.load(f)
    print(f'Resuming - {len(inner_results)} runs already complete.')
else:
    inner_results = {}

# creates inner split
def get_inner_split(outer_train, inner_fold_dict):
    inner_train = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_train']]
    inner_val   = [g for g in outer_train if g.subject_id in inner_fold_dict['inner_val']]
    return inner_train, inner_val

# the following loop performs the hyperparamter tuning train/val phase
# it iterates over all outer and inner folds, training/validating all configurations
outer_bar = tqdm(range(N_OUTER), desc='Outer folds', position=0)
for outer_idx in outer_bar:
    outer_train = outer_folds[outer_idx]['train']

    inner_bar = tqdm(range(N_INNER), desc=f'  Inner folds', position=1, leave=False)
    for inner_idx in inner_bar:
        inner_train, inner_val = get_inner_split(outer_train, inner_fold_ids[outer_idx][inner_idx])
        train_labels = get_labels(inner_train)
        train_loader = DataLoader(inner_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(inner_val,   batch_size=BATCH, shuffle=False)

        # Calculate degree distribution for PNAConv from the training dataset
        degrees = []
        for data in inner_train:
            if data.num_nodes > 0:
                # degree expects edge_index[1] for target nodes
                degrees.append(degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long))

        if degrees: # Check if list of degrees is not empty
            all_degrees = torch.cat(degrees) # Concatenate all degrees
            max_degree = int(all_degrees.max()) if all_degrees.numel() > 0 else 0
        else:
            max_degree = 0 # No nodes or no edges

        # Ensure minlength for torch.bincount is at least 1
        min_len_bincount = max_degree + 1
        if min_len_bincount == 0: # Handle cases where max_degree could be -1 (empty tensor) or 0
            min_len_bincount = 1 # Must be at least 1 for bincount

        deg = torch.zeros(min_len_bincount, dtype=torch.long)
        for data in inner_train:
            if data.num_nodes > 0:
                d = degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long)
                deg += torch.bincount(d, minlength=min_len_bincount)

        cfg_bar = tqdm(range(len(CONFIGS)), desc='    Configs', position=2, leave=False)
        for cfg_idx in cfg_bar:
            key = (outer_idx, inner_idx, cfg_idx)
            if key in inner_results:
                cfg_bar.set_postfix_str('skipped')
                continue

            cfg           = CONFIGS[cfg_idx]
            criterion     = build_criterion(cfg, train_labels)
            model, opt    = build_model_and_opt(cfg, deg) # Pass deg here
            _, metrics    = train_with_early_stopping(model, opt, criterion, train_loader, val_loader, deg) # Pass deg here

            inner_results[key] = metrics
            cfg_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                                recall_mci=f'{metrics["recall_mci"]:.3f}')

        # save after every inner fold
        with open(INNER_RESULTS_PATH, 'wb') as f:
            pickle.dump(inner_results, f)

print('Hyperparameter tuning comlpeted.')

# average metrics per config across all 15 runs
inner_avg = {}
for cfg_idx in range(len(CONFIGS)):
    runs = [inner_results[(o, i, cfg_idx)] for o in range(N_OUTER) for i in range(N_INNER)]
    inner_avg[cfg_idx] = avg_metrics(runs)

print('\nInner-loop averaged metrics per config:')
df_inner = pd.DataFrame([
    {'config': i, **inner_avg[i], **CONFIGS[i]}
    for i in range(len(CONFIGS))
]).set_index('config')
print(df_inner.to_string())
df_inner.to_csv(os.path.join(OUTPUT_DIR, 'inner_avg_metrics.csv'))

Resuming - 540 runs already complete.


  Inner folds:   0%|          | 0/3 [00:00<?, ?it/s]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|          | 0/36 [00:00<?, ?it/s, skipped]

    Configs:   0%|       

Hyperparameter tuning comlpeted.

Inner-loop averaged metrics per config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma      wd  num_layers
config                                                                                                                                                           
0           0.716683  0.705837    0.733601   0.699764       0.389616      0.918516  0.498374  0.785980  0.642177  0.756903         3.5    1.0  0.0010           2
1           0.731843  0.715932    0.759185   0.704501       0.398310      0.924263  0.516017  0.796326  0.656171  0.760411         3.5    1.0  0.0010           3
2           0.724528  0.723912    0.727616   0.721439       0.402667      0.919355  0.509552  0.803246  0.656399  0.757845         3.5    1.0  0.0001           2
3           0.723522  0.723851    0.728384   0.718660       0.407794      0.919343  0.513800  0.801931  0.657866  0.

In [ ]:
INNER_FILTER = {
    'balanced_acc': 0.65,
    'recall_mci':   0.75,
    'recall_cn':    0.55,
}

surviving_configs = [
    i for i in range(len(CONFIGS))
    if passes_filter(inner_avg[i], INNER_FILTER)
]

print(f'Configs surviving inner filter: {len(surviving_configs)}/{len(CONFIGS)}')
for i in surviving_configs:
    print(f'  [{i:02d}] bal_acc={inner_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={inner_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={inner_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

Configs surviving inner filter: 7/18
  [00] bal_acc=0.710  recall_mci=0.804  recall_cn=0.616  |  {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.001}
  [01] bal_acc=0.712  recall_mci=0.824  recall_cn=0.599  |  {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.0001}
  [06] bal_acc=0.710  recall_mci=0.827  recall_cn=0.593  |  {'mci_weight': 4.0, 'gamma': 1.0, 'wd': 0.001}
  [07] bal_acc=0.712  recall_mci=0.827  recall_cn=0.597  |  {'mci_weight': 4.0, 'gamma': 1.0, 'wd': 0.0001}
  [12] bal_acc=0.703  recall_mci=0.851  recall_cn=0.555  |  {'mci_weight': 4.5, 'gamma': 1.0, 'wd': 0.001}
  [13] bal_acc=0.711  recall_mci=0.806  recall_cn=0.615  |  {'mci_weight': 4.5, 'gamma': 1.0, 'wd': 0.0001}
  [15] bal_acc=0.697  recall_mci=0.844  recall_cn=0.551  |  {'mci_weight': 4.5, 'gamma': 1.5, 'wd': 0.0001}


In [ ]:
# surviving configurations get trained on the full train set, and validated on the outer val set
CKPT_DIR          = os.path.join(OUTPUT_DIR, 'outer_checkpoints')
OUTER_VAL_RESULTS_PATH = os.path.join(OUTPUT_DIR, 'outer_val_results.pkl')
os.makedirs(CKPT_DIR, exist_ok=True)

# resume if training stopped midway
if os.path.exists(OUTER_VAL_RESULTS_PATH):
    with open(OUTER_VAL_RESULTS_PATH, 'rb') as f:
        outer_val_results = pickle.load(f)
    print(f'Resuming — {len(outer_val_results)} runs already complete.')
else:
    outer_val_results = {}

# the following loop iterates over all outer folds, for all configurations
# each configuration gets trained on the whole train set and validated
# on the outer val set of each fold
cfg_bar = tqdm(surviving_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc=f'  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        key = (cfg_idx, outer_idx)
        if key in outer_val_results:
            outer_bar.set_postfix_str('skipped')
            continue

        outer_train  = outer_folds[outer_idx]['train']
        outer_val    = outer_folds[outer_idx]['val']
        train_labels = get_labels(outer_train)
        train_loader = DataLoader(outer_train, batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(outer_val,   batch_size=BATCH, shuffle=False)

        # Calculate degree distribution for PNAConv from the training dataset
        degrees = []
        for data in outer_train:
            if data.num_nodes > 0:
                degrees.append(degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long))

        if degrees:
            all_degrees = torch.cat(degrees)
            max_degree = int(all_degrees.max()) if all_degrees.numel() > 0 else 0
        else:
            max_degree = 0

        min_len_bincount = max_degree + 1
        if min_len_bincount == 0:
            min_len_bincount = 1

        deg = torch.zeros(min_len_bincount, dtype=torch.long)
        for data in outer_train:
            if data.num_nodes > 0:
                d = degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long)
                deg += torch.bincount(d, minlength=min_len_bincount)

        criterion    = build_criterion(cfg, train_labels)
        model, opt   = build_model_and_opt(cfg, deg) # Pass deg here
        model, metrics = train_with_early_stopping(model, opt, criterion, train_loader, val_loader, deg) # Pass deg here

        outer_val_results[key] = metrics
        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        torch.save(model.state_dict(), ckpt_path)

        with open(OUTER_VAL_RESULTS_PATH, 'wb') as f:
            pickle.dump(outer_val_results, f)

print('Training and validation completed.')

# average outer val metrics per config across 5 folds
outer_val_avg = {
    i: avg_metrics([outer_val_results[(i, o)] for o in range(N_OUTER)])
    for i in surviving_configs
}

print('\nOuter-val averaged metrics per surviving config:')
df_val = pd.DataFrame([
    {'config': i, **outer_val_avg[i], **CONFIGS[i]}
    for i in surviving_configs
]).set_index('config')
print(df_val.to_string())
df_val.to_csv(os.path.join(OUTPUT_DIR, 'outer_val_avg_metrics.csv'))

Configs:   0%|          | 0/7 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

Training and validation completed.

Outer-val averaged metrics per surviving config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma      wd
config                                                                                                                                               
0           0.753640  0.716883    0.815000   0.692279       0.404785      0.942665  0.532443  0.794037  0.663240  0.782480         3.5    1.0  0.0010
1           0.744098  0.693506    0.828333   0.659863       0.376558      0.941395  0.516367  0.774049  0.645208  0.790508         3.5    1.0  0.0001
6           0.747212  0.680519    0.856667   0.637758       0.375665      0.948494  0.518742  0.758352  0.638547  0.807501         4.0    1.0  0.0010
7           0.775818  0.703896    0.895000   0.656637       0.400916      0.960339  0.551451  0.776821  0.664136  0.835026         4.0    1.0  0.0001
12          0.7

In [ ]:
# filtering models based on performance on outer val
OUTER_VAL_FILTER = {
    'balanced_acc': 0.70,
    'recall_mci':   0.75,
    'recall_cn':    0.65,
}

final_configs = [
    i for i in surviving_configs
    if passes_filter(outer_val_avg[i], OUTER_VAL_FILTER)
]

print(f'Configs surviving outer val filter: {len(final_configs)}/{len(surviving_configs)}')
for i in final_configs:
    print(f'  [{i:02d}] bal_acc={outer_val_avg[i]["balanced_acc"]:.3f}  '
          f'recall_mci={outer_val_avg[i]["recall_mci"]:.3f}  '
          f'recall_cn={outer_val_avg[i]["recall_cn"]:.3f}  |  {CONFIGS[i]}')

if not final_configs:
    raise RuntimeError('No configs survived. Loosen OUTER_VAL_FILTER thresholds.')

Configs surviving outer val filter: 4/7
  [00] bal_acc=0.754  recall_mci=0.815  recall_cn=0.692  |  {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.001}
  [01] bal_acc=0.744  recall_mci=0.828  recall_cn=0.660  |  {'mci_weight': 3.5, 'gamma': 1.0, 'wd': 0.0001}
  [07] bal_acc=0.776  recall_mci=0.895  recall_cn=0.657  |  {'mci_weight': 4.0, 'gamma': 1.0, 'wd': 0.0001}
  [12] bal_acc=0.751  recall_mci=0.842  recall_cn=0.660  |  {'mci_weight': 4.5, 'gamma': 1.0, 'wd': 0.001}


In [ ]:
# testing
test_results = {i: {} for i in final_configs}

# the following loop tests all configurations on the test set of all 5 outer folds
cfg_bar = tqdm(final_configs, desc='Configs', position=0)
for cfg_idx in cfg_bar:
    cfg = CONFIGS[cfg_idx]

    outer_bar = tqdm(range(N_OUTER), desc='  Outer folds', position=1, leave=False)
    for outer_idx in outer_bar:
        ckpt_path = os.path.join(CKPT_DIR, f'cfg{cfg_idx:02d}_outer{outer_idx+1}.pt')
        model, _  = build_model_and_opt(cfg)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        test_loader          = DataLoader(outer_folds[outer_idx]['test'], batch_size=BATCH, shuffle=False)
        labels, preds, probs = evaluate(model, test_loader)
        metrics              = compute_metrics(labels, preds, probs)

        test_results[cfg_idx][outer_idx] = {
            'metrics': metrics,
            'probs':   probs,
            'preds':   preds,
            'labels':  labels,
        }

        outer_bar.set_postfix(bal_acc=f'{metrics["balanced_acc"]:.3f}',
                              recall_mci=f'{metrics["recall_mci"]:.3f}',
                              recall_cn=f'{metrics["recall_cn"]:.3f}')

# per-config average across outer folds
test_avg = {
    i: avg_metrics([test_results[i][o]['metrics'] for o in range(N_OUTER)])
    for i in final_configs
}

print('\nTest averaged metrics per final config:')
df_test = pd.DataFrame([
    {'config': i, **test_avg[i], **CONFIGS[i]}
    for i in final_configs
]).set_index('config')
print(df_test.to_string())
df_test.to_csv(os.path.join(OUTPUT_DIR, 'test_avg_metrics.csv'))

Configs:   0%|          | 0/4 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]


Test averaged metrics per final config:
        balanced_acc  accuracy  recall_mci  recall_cn  precision_mci  precision_cn    f1_mci     f1_cn  f1_macro       auc  mci_weight  gamma      wd
config                                                                                                                                               
0           0.654736  0.606299    0.735023   0.574449       0.304424      0.900136  0.427208  0.698414  0.562811  0.723357         3.5    1.0  0.0010
1           0.677919  0.618898    0.774934   0.580905       0.319069      0.914952  0.448745  0.706558  0.577651  0.742046         3.5    1.0  0.0001
7           0.662426  0.614173    0.742461   0.582392       0.309135      0.901518  0.434978  0.706987  0.570983  0.742227         4.0    1.0  0.0001
12          0.665030  0.596850    0.777373   0.552687       0.304708      0.911023  0.434511  0.684344  0.559427  0.737353         4.5    1.0  0.0010


In [ ]:
# ensemble per outer fold
# current strat: weighted average of class probabilities,
# weight = outer val balanced accuracy of that model on that fold.

print('Ensemble results per outer fold:')
ensemble_results = {}

for outer_idx in range(N_OUTER):
    labels         = test_results[final_configs[0]][outer_idx]['labels']
    n              = len(labels)
    weighted_probs = np.zeros((n, 2))
    total_weight   = 0.0

    for cfg_idx in final_configs:
        weight          = outer_val_results[(cfg_idx, outer_idx)]['balanced_acc']
        weighted_probs += weight * test_results[cfg_idx][outer_idx]['probs']
        total_weight   += weight

    weighted_probs /= total_weight
    ensemble_preds  = weighted_probs.argmax(axis=1)
    metrics         = compute_metrics(labels, ensemble_preds, weighted_probs)

    ensemble_results[outer_idx] = {
        **metrics,
        'labels': labels,
        'preds':  ensemble_preds,
        'probs':  weighted_probs,
    }

    print(f'  Outer fold {outer_idx+1}: '
          f'bal_acc={metrics["balanced_acc"]:.3f}  '
          f'recall_mci={metrics["recall_mci"]:.3f}  '
          f'recall_cn={metrics["recall_cn"]:.3f}  '
          f'auc={metrics["auc"]:.3f}')

ensemble_avg = avg_metrics([
    {k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}
    for o in range(N_OUTER)
])
print(f'\nEnsemble average across folds:')
for k, v in ensemble_avg.items():
    print(f'  {k}: {v:.4f}')

df_ensemble = pd.DataFrame([
    {'outer_fold': o+1, **{k: v for k, v in ensemble_results[o].items() if k not in ('labels', 'preds', 'probs')}}
    for o in range(N_OUTER)
] + [{'outer_fold': 'avg', **ensemble_avg}])
df_ensemble.to_csv(os.path.join(OUTPUT_DIR, 'ensemble_results.csv'), index=False)
print(f'\nAll results saved to {OUTPUT_DIR}')

Ensemble results per outer fold:
  Outer fold 1: bal_acc=0.660  recall_mci=0.864  recall_cn=0.457  auc=0.742
  Outer fold 2: bal_acc=0.600  recall_mci=0.680  recall_cn=0.520  auc=0.680
  Outer fold 3: bal_acc=0.727  recall_mci=0.857  recall_cn=0.596  auc=0.842
  Outer fold 4: bal_acc=0.599  recall_mci=0.625  recall_cn=0.573  auc=0.678
  Outer fold 5: bal_acc=0.739  recall_mci=0.778  recall_cn=0.700  auc=0.780

Ensemble average across folds:
  balanced_acc: 0.6649
  accuracy: 0.6063
  recall_mci: 0.7607
  recall_cn: 0.5691
  precision_mci: 0.3097
  precision_cn: 0.9070
  f1_mci: 0.4366
  f1_cn: 0.6959
  f1_macro: 0.5663
  auc: 0.7444

All results saved to /content/drive/MyDrive/team3_xai_gnn/transformer_results


In [ ]:
from sklearn.metrics import confusion_matrix

for outer_idx in range(N_OUTER):
    print(f'=== Outer Fold {outer_idx+1} ===')

    for cfg_idx in final_configs:
        r  = test_results[cfg_idx][outer_idx]
        cm = confusion_matrix(r['labels'], r['preds'])
        print(f'cfg{cfg_idx} (mw={CONFIGS[cfg_idx]["mci_weight"]} g={CONFIGS[cfg_idx]["gamma"]} wd={CONFIGS[cfg_idx]["wd"]})')
        print(f'{"":10s}  Pred CN  Pred MCI')
        print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
        print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}\n')

    er = ensemble_results[outer_idx]
    cm = confusion_matrix(er['labels'], er['preds'])
    print(f'ENSEMBLE')
    print(f'{"":10s}  Pred CN  Pred MCI')
    print(f'  True CN   {cm[0,0]:4d}     {cm[0,1]:4d}')
    print(f'  True MCI  {cm[1,0]:4d}     {cm[1,1]:4d}')
    print()

=== Outer Fold 1 ===
cfg0 (mw=3.5 g=1.0 wd=0.001)
            Pred CN  Pred MCI
  True CN     54       51
  True MCI     4       18

cfg1 (mw=3.5 g=1.0 wd=0.0001)
            Pred CN  Pred MCI
  True CN     45       60
  True MCI     3       19

cfg7 (mw=4.0 g=1.0 wd=0.0001)
            Pred CN  Pred MCI
  True CN     59       46
  True MCI     5       17

cfg12 (mw=4.5 g=1.0 wd=0.001)
            Pred CN  Pred MCI
  True CN     54       51
  True MCI     3       19

ENSEMBLE
            Pred CN  Pred MCI
  True CN     48       57
  True MCI     3       19

=== Outer Fold 2 ===
cfg0 (mw=3.5 g=1.0 wd=0.001)
            Pred CN  Pred MCI
  True CN     58       44
  True MCI     9       16

cfg1 (mw=3.5 g=1.0 wd=0.0001)
            Pred CN  Pred MCI
  True CN     57       45
  True MCI     9       16

cfg7 (mw=4.0 g=1.0 wd=0.0001)
            Pred CN  Pred MCI
  True CN     57       45
  True MCI     7       18

cfg12 (mw=4.5 g=1.0 wd=0.001)
            Pred CN  Pred MCI
  True CN     46 